# QNG Euclidean Baseline -- Results Analysis

This notebook loads experiment results and produces comparison plots across all tasks.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
from src.metrics import load_results
from src.visualization import convergence_plot, resource_plot, walltime_plot, final_loss_bar, LABELS

## Load Results

In [ ]:
results_dir = os.path.join('..', 'results')

fit1d = load_results(os.path.join(results_dir, 'function_fitting_1d.json'))
fit2d = load_results(os.path.join(results_dir, 'function_fitting_2d.json'))
vqe  = load_results(os.path.join(results_dir, 'vqe.json'))
cls  = load_results(os.path.join(results_dir, 'classification.json'))

# VQE has a 'config' key; separate it
vqe_config = vqe.pop('config', {})
print('VQE config:', vqe_config)

## 1. Function Fitting (1D): sin(x)

In [ ]:
convergence_plot(fit1d, title='1D Regression: sin(x)', log_y=True)
plt.show()

resource_plot(fit1d, title='1D Regression: sin(x)  (resource-normalised)', log_y=True)
plt.show()

final_loss_bar(fit1d, title='1D Regression: final MSE')
plt.show()

## 2. Function Fitting (2D): (x1²+x2²)/2

In [ ]:
convergence_plot(fit2d, title='2D Regression: (x1\u00b2+x2\u00b2)/2', log_y=True)
plt.show()

resource_plot(fit2d, title='2D Regression (resource-normalised)', log_y=True)
plt.show()

final_loss_bar(fit2d, title='2D Regression: final MSE')
plt.show()

## 3. VQE: Transverse-Field Ising Model

In [ ]:
E_exact = vqe_config.get('E_exact', None)
title_vqe = f'VQE Ising 4q  (E*={E_exact:.3f})' if E_exact else 'VQE Ising 4q'

fig, ax = convergence_plot(vqe, title=title_vqe, ylabel='Energy \u27e8H\u27e9')
if E_exact:
    ax.axhline(E_exact, color='k', linestyle='--', alpha=0.5, label=f'Exact = {E_exact:.3f}')
    ax.legend()
plt.show()

resource_plot(vqe, title='VQE (resource-normalised)', ylabel='Energy \u27e8H\u27e9')
plt.show()

final_loss_bar(vqe, title='VQE: final energy', ylabel='Energy \u27e8H\u27e9')
plt.show()

## 4. Classification: make_moons

In [ ]:
convergence_plot(cls, title='Classification (make_moons)', log_y=True)
plt.show()

resource_plot(cls, title='Classification (resource-normalised)', log_y=True)
plt.show()

final_loss_bar(cls, title='Classification: final hinge loss')
plt.show()

## 5. Summary Table

In [ ]:
opt_names = ['GD', 'Adam', 'QNG_block', 'QNG_full']
tasks = {
    '1D Regression': fit1d,
    '2D Regression': fit2d,
    'VQE': vqe,
    'Classification': cls,
}

print(f'{"Task":<20}', end='')
for o in opt_names:
    print(f'{LABELS[o]:>20}', end='')
print()
print('-' * (20 + 20 * len(opt_names)))

for task_name, data in tasks.items():
    print(f'{task_name:<20}', end='')
    for o in opt_names:
        m = data[o]['final_loss_mean']
        s = data[o]['final_loss_std']
        print(f'{m:>14.4f} \u00b1{s:.4f}', end='')
    print()

# Classification accuracy
if 'final_acc_mean' in cls.get('GD', {}):
    print(f'\n{"Cls Accuracy":<20}', end='')
    for o in opt_names:
        m = cls[o].get('final_acc_mean', float('nan'))
        s = cls[o].get('final_acc_std', float('nan'))
        print(f'{m:>14.2%} \u00b1{s:.2%}', end='')
    print()